In [1]:
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import requests
from io import BytesIO

# Direct download of pre-trained model + scaler (hosted on Hugging Face)
MODEL_URL = "https://huggingface.co/spaces/akhaliq/diabetes-prediction/resolve/main/diabetes_model.pkl"
SCALER_URL = "https://huggingface.co/spaces/akhaliq/diabetes-prediction/resolve/main/scaler.pkl"

def load_model_from_url(url):
    response = requests.get(url)
    response.raise_for_status()
    return joblib.load(BytesIO(response.content))

# Load model and scaler (cached after first request)
try:
    model = load_model_from_url(MODEL_URL)
    scaler = load_model_from_url(SCALER_URL)
except Exception as e:
    # Fallback: train a simple model on the fly (fast, ~5 sec)
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    import pandas as pd

    df = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv",
                     names=["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
                            "BMI","DiabetesPedigreeFunction","Age","Outcome"])
    # Clean zeros
    for col in ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]:
        median_val = df[df[col]!=0][col].median()
        df[col] = df[col].replace(0, median_val)
    X = df.drop("Outcome", axis=1)
    y = df["Outcome"]
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

def predict_diabetes(pregnancies, glucose, blood_pressure, skin_thickness,
                     insulin, bmi, dpf, age):
    features = np.array([[pregnancies, glucose, blood_pressure,
                          skin_thickness, insulin, bmi, dpf, age]])
    scaled = scaler.transform(features)
    proba = model.predict_proba(scaled)[0][1]
    pred = "Diabetic" if proba >= 0.5 else "Non-diabetic"
    return f"**Prediction:** {pred}\n\n**Confidence:** {proba:.1%}"

inputs = [
    gr.Number(label="Pregnancies", value=2),
    gr.Number(label="Glucose", value=120),
    gr.Number(label="Blood Pressure", value=70),
    gr.Number(label="Skin Thickness", value=25),
    gr.Number(label="Insulin", value=80),
    gr.Number(label="BMI", value=25.0),
    gr.Number(label="Diabetes Pedigree Function", value=0.5),
    gr.Number(label="Age", value=30)
]

gr.Interface(
    fn=predict_diabetes,
    inputs=inputs,
    outputs=gr.Markdown(),

    title=" Diabetes Risk Predictor",
    description="Enter your health metrics to get an instant prediction.",
    examples=[
        [2, 90, 70, 20, 80, 25.0, 0.5, 30],
        [5, 160, 85, 35, 200, 35.0, 0.9, 55]
    ]
).launch(share=True)

c:\Users\shahz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://94f4aeebae7800eb18.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


c:\Users\shahz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\shahz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\shahz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
